In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import FeatureUnion

In [2]:
df = pd.read_csv('../After_EDA_data/Light_text.csv')

In [3]:
df

,uid,profile,anime_uid,score,scores,text
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...
...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192112 entries, 0 to 192111
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        192112 non-null  int64 
 1   profile    192112 non-null  object
 2   anime_uid  192112 non-null  int64 
 3   score      192112 non-null  int64 
 4   scores     192112 non-null  object
 5   text       192112 non-null  object
dtypes: int64(3), object(3)
memory usage: 8.8+ MB


In [5]:
df.shape

(192112, 6)

In [6]:
def classification(text):
    if text > 6:
        return 'Good'
    elif text > 4:
        return 'Neutral'
    else:
        return 'Bad'

In [7]:
df['target'] = df['score'].apply(classification)

In [8]:
df['target']

0         Good
1         Good
2         Good
3         Good
4         Good
          ... 
192107    Good
192108    Good
192109     Bad
192110    Good
192111    Good
Name: target, Length: 192112, dtype: object

In [9]:
df['target'].value_counts()

target
Good       142343
Neutral     27651
Bad         22118
Name: count, dtype: int64

In [10]:
df_majority = df[df['target'] == 'Good']
df_minority = df[df['target'] == 'Bad']
df_neutral = df[df['target'] == "Neutral"]

In [11]:
df_reduced = df_majority.sample(n=len(df_minority), random_state=42)

In [12]:
df_balanced = pd.concat([df_reduced, df_minority, df_neutral]).sample(frac=1, random_state=42).reset_index(drop=True)

In [13]:
df_balanced

,uid,profile,anime_uid,score,scores,text,target
0,280316,QuakAtak,35413,8,"{'Overall': '8', 'Story': '7', 'Animation': '6...",there s something satisfying about a show that...,Good
1,236166,LackOfARealLife,30276,10,"{'Overall': '10', 'Story': '8', 'Animation': '...",it takes an unbelievable amount of skill to us...,Good
2,252395,xero657,34051,8,"{'Overall': '8', 'Story': '7', 'Animation': '8...",so i was originally gonna pass up this anime i...,Good
3,143099,vigorousjammer,6616,6,"{'Overall': '6', 'Story': '7', 'Animation': '5...",coffee samurai is a 30 minute short film about...,Neutral
4,310015,No_Honor,20785,6,"{'Overall': '6', 'Story': '7', 'Animation': '7...",i don t really know how to write a review so i...,Neutral
...,...,...,...,...,...,...,...
71882,132657,FullmetalCowboy,5957,3,"{'Overall': '3', 'Story': '2', 'Animation': '5...",critic s log earthdate february 14 2013 supple...,Bad
71883,141887,Flueckli,10098,7,"{'Overall': '7', 'Story': '6', 'Animation': '4...",don t expect too much out of these shortstorie...,Good
71884,299523,The_McChicken,37786,5,"{'Overall': '5', 'Story': '9', 'Animation': '7...",story is very good and i feel pretty realistic...,Neutral
71885,18387,boundtotomorrow,853,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",to put it simply i wish more if not all animes...,Good


In [14]:
df_balanced['target'].value_counts()

target
Neutral    27651
Good       22118
Bad        22118
Name: count, dtype: int64

In [15]:
x = df_balanced['text']
y = df_balanced['target']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

In [17]:
word_tfidf = TfidfVectorizer(
    max_features=8000,
    min_df=10,
    max_df=0.9,
    sublinear_tf=True,
    ngram_range=(1, 2)
)

In [18]:
char_tfidf = TfidfVectorizer(
    max_features=8000,
    analyzer='char_wb',
    min_df=10,
    max_df=0.9,
    ngram_range=(3, 5),
    sublinear_tf=True
)

In [19]:
tfidf = FeatureUnion([
    ('word', word_tfidf),
    ('char', char_tfidf)
])

In [20]:
# tfidf = TfidfVectorizer(
#     max_features=5000,
#     sublinear_tf=True,
#     ngram_range=(1, 2)
# )

In [21]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [22]:
model = MultinomialNB(
    alpha=1
)

model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [23]:
y_pred = model.predict(X_test_tfidf)

In [24]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         Bad       0.68      0.64      0.66      4424
        Good       0.80      0.74      0.77      4424
     Neutral       0.61      0.67      0.64      5530

    accuracy                           0.68     14378
   macro avg       0.69      0.68      0.69     14378
weighted avg       0.69      0.68      0.68     14378



In [25]:
print(confusion_matrix(y_test, y_pred))

[[2818  143 1463]
 [ 215 3265  944]
 [1133  686 3711]]
